In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio torch ultralytics pydantic opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.2 MB/s eta 0:00:00


In [ ]:
import subprocess
subprocess.run(['ngrok', 'kill'])
print('All active ngrok sessions killed.')

All active ngrok sessions killed.


In [ ]:
import os
import threading
import asyncio
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
import torch
from ultralytics import YOLO
import base64
import numpy as np
import cv2

# --- 1. THE SELF-HEALING PROTOCOL ---
print("🧹 Purging old network tunnels and freeing Port 8000...")
ngrok.kill()
os.system("fuser -k 8000/tcp") # Force kill any zombie Uvicorn servers!

# Initialize the FastAPI application
app = FastAPI(title="TitanOPS AIR - Neural Brain API")

# Load the YOLO model onto the T4 GPU
print("🧠 IGNITING T4 GPU. LOADING NEURAL NETWORK...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolo11n.pt')
model.to(device)
print(f"✅ VISION PIPELINE ONLINE. RUNNING ON: {device.upper()}")

# Define the expected data structure from the Codespace
class SensorPayload(BaseModel):
    agent_id: str
    timestamp: float
    image_b64: str

@app.post("/process_vision")
async def process_vision(payload: SensorPayload):
    try:
        img_data = base64.b64decode(payload.image_b64)
        np_arr = np.frombuffer(img_data, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        if frame is None:
            raise ValueError("Failed to decode image.")

        results = model(frame, conf=0.4, verbose=False)

        detections = []
        for result in results:
            boxes = result.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf[0])
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name in ['person', 'car', 'truck']:
                    detections.append({
                        "type": class_name,
                        "confidence": round(conf, 3),
                        "bounding_box": [int(x1), int(y1), int(x2), int(y2)],
                        "center_pixel": [int((x1 + x2) / 2), int((y1 + y2) / 2)]
                    })
                    # Draw the red bounding box and label directly on the frame!
                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 3)
                    cv2.putText(frame, f"THREAT: {class_name.upper()}", (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        status = "INTERCEPT" if len(detections) > 0 else "PATROL"

        return {
            "status": "SUCCESS",
            "agent_id": payload.agent_id,
            "swarm_command": status,
            "threats_detected": len(detections),
            "threat_data": detections
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- 2. SECURE TUNNEL INITIALIZATION ---
# REPLACE THE STRING BELOW WITH YOUR ACTUAL NGROK TOKEN!
NGROK_AUTH_TOKEN = "2z0fiTD4ac10yHsF1eD9uzAkmbq_7HybYrKf8TVGwTPw2Tru1"

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"\n==================================================")
    print(f"🔗 TACTICAL C2 UPLINK SECURED.")
    print(f"📡 TITAN_OPS URL: {public_url.public_url}")
    print(f"==================================================\n")
except Exception as e:
    print(f"❌ Ngrok Tunnel Failed. Check your Auth Token. Error: {e}")

# --- 3. BACKGROUND SERVER LAUNCH ---
def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

print("🚀 Launching Background Microservice...")
thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("✅ Server is actively listening in the background. Ready for Codespace telemetry.")

KeyboardInterrupt: 

In [ ]:
import os
import threading
import asyncio
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
import torch
from ultralytics import YOLO
import base64
import numpy as np
import cv2

# --- 1. THE SELF-HEALING PROTOCOL ---
print("🧹 Purging old network tunnels and freeing Port 8000...")
ngrok.kill()
os.system("fuser -k 8000/tcp")

app = FastAPI(title="TitanOPS AIR - Neural Brain API")

# Load the YOLO model onto the T4 GPU
print("🧠 IGNITING T4 GPU. LOADING NEURAL NETWORK...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = YOLO('yolo11n.pt')
model.to(device)
print(f"✅ VISION PIPELINE ONLINE. RUNNING ON: {device.upper()}")

class SensorPayload(BaseModel):
    agent_id: str
    timestamp: float
    image_b64: str

@app.post("/process_vision")
async def process_vision(payload: SensorPayload):
    try:
        img_data = base64.b64decode(payload.image_b64)
        np_arr = np.frombuffer(img_data, np.uint8)
        frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

        if frame is None: raise ValueError("Failed to decode image.")

        results = model(frame, conf=0.4, verbose=False)
        detections = []

        for result in results:
            boxes = result.boxes
            for box in boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf[0])
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name in ['person', 'car', 'truck']:
                    # 🎨 NEW: Draw the Tactical Red Bounding Boxes on the GPU!
                    cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 3)
                    cv2.putText(frame, f"THREAT: {class_name.upper()} [{int(conf*100)}%]", (int(x1), int(y1)-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

                    detections.append({
                        "type": class_name,
                        "confidence": round(conf, 3),
                        "bounding_box": [int(x1), int(y1), int(x2), int(y2)],
                        "center_pixel": [int((x1 + x2) / 2), int((y1 + y2) / 2)]
                    })

        # 📸 NEW: Encode the modified image back to Base64 to send to the UI
        _, buffer = cv2.imencode('.jpg', frame)
        processed_b64 = base64.b64encode(buffer).decode('utf-8')

        status = "INTERCEPT" if len(detections) > 0 else "PATROL"

        return {
            "status": "SUCCESS",
            "agent_id": payload.agent_id,
            "swarm_command": status,
            "threats_detected": len(detections),
            "threat_data": detections,
            "processed_image_b64": processed_b64 # Included in payload
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# --- 2. SECURE TUNNEL INITIALIZATION ---
NGROK_AUTH_TOKEN = "2z0fiTD4ac10yHsF1eD9uzAkmbq_7HybYrKf8TVGwTPw2Tru1"

try:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(8000)
    print(f"\n==================================================")
    print(f"🔗 TACTICAL C2 UPLINK SECURED.")
    print(f"📡 TITAN_OPS URL: {public_url.public_url}")
    print(f"==================================================\n")
except Exception as e:
    print(f"❌ Ngrok Tunnel Failed. Check your Auth Token. Error: {e}")

# --- 3. BACKGROUND SERVER LAUNCH ---
def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="error")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

print("🚀 Launching Background Microservice...")
thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("✅ Server is actively listening in the background. Ready for Codespace telemetry.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
🧹 Purging old network tunnels and freeing Port 8000...
🧠 IGNITING T4 GPU. LOADING NEURAL NETWORK...
✅ VISION PIPELINE ONLINE. RUNNING ON: CPU

🔗 TACTICAL C2 UPLINK SECURED.
📡 TITAN_OPS URL: https://ed5d-34-24-185-230.ngrok-free.app

🚀 Launching Background Microservice...
✅ Server is actively listening in the background. Ready for Codespace telemetry.
